# INIA — Segmentation API Testing & Comparison

This notebook tests the unified API, runs the 3-architecture comparison, and exports bounding boxes for Team 2.

## 0. Install Dependencies

In [ ]:
!pip install keras_unet_collection -q

## 0.1 Upload INIA.py

Upload the `INIA.py` file to the Colab working directory. Run the cell below — it will open a file picker.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select INIA.py from your computer

## 0.2 Download Dataset

In [ ]:
!curl -L -o images.npz https://github.com/DivyanshuTak/Ultrasoud_Unet_Segmentation/raw/refs/heads/main/images.npz
!curl -L -o masks.npz https://github.com/DivyanshuTak/Ultrasoud_Unet_Segmentation/raw/refs/heads/main/masks.npz

---
## Step 1 — Test the API with UNet++

This should reproduce your original notebook results (Dice ~0.94 on test).

In [ ]:
from INIA import load_data, fit, evaluate, plot_history, plot_predictions

# Load and preprocess data
X_train, y_train, X_test, y_test = load_data()

In [ ]:
# Train UNet++ (your original architecture)
model_pp, history_pp = fit("unet++", X_train, y_train, epochs=50)

In [ ]:
# Evaluate on test set
results_pp = evaluate(model_pp, X_test, y_test)

In [ ]:
# Plot training curves
plot_history(history_pp, "UNet++")

In [ ]:
# Visual check — see predictions vs ground truth
plot_predictions(model_pp, X_test, y_test, n=3)

### ✅ Checkpoint
Does the test Dice look close to **0.94**? If yes, move on. If not, debug before continuing.

---
## Step 2 — Run 3-Architecture Comparison

This trains UNet, UNet++, and UNet3++ under identical conditions and produces the comparison table.

In [ ]:
from INIA import compare
import pandas as pd

results_df, all_histories = compare(
    X_train, y_train, X_test, y_test,
    architectures=["unet", "unet++", "unet3++"],
    epochs=50
)

In [ ]:
# Display comparison table
results_df

### Comparison Plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Bar chart: Dice & IoU per architecture ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

archs = results_df.index.tolist()
x = np.arange(len(archs))
width = 0.5

# Dice
axes[0].bar(x, results_df['dice_coef'], width, color=['#2196F3', '#FF9800', '#4CAF50'])
axes[0].set_title('Test Dice by Architecture', fontsize=14)
axes[0].set_ylabel('Dice Coefficient')
axes[0].set_xticks(x)
axes[0].set_xticklabels(archs)
axes[0].set_ylim(0.8, 1.0)
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(results_df['dice_coef']):
    axes[0].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

# IoU
axes[1].bar(x, results_df['iou_coef'], width, color=['#2196F3', '#FF9800', '#4CAF50'])
axes[1].set_title('Test IoU by Architecture', fontsize=14)
axes[1].set_ylabel('IoU Coefficient')
axes[1].set_xticks(x)
axes[1].set_xticklabels(archs)
axes[1].set_ylim(0.7, 1.0)
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(results_df['iou_coef']):
    axes[1].text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# --- Training curves: Dice over epochs for each architecture ---
fig, ax = plt.subplots(figsize=(10, 6))

colors = {'unet': '#2196F3', 'unet++': '#FF9800', 'unet3++': '#4CAF50'}

for arch, hist in all_histories.items():
    dice_vals = hist.history.get('val_dice_coef', hist.history.get('dice_coef', []))
    ax.plot(dice_vals, label=arch, linewidth=2, color=colors.get(arch, None))

ax.set_title('Validation Dice Over Epochs', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Dice Coefficient')
ax.set_ylim(0, 1.0)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Step 3 — Export Bounding Boxes for Team 2 (SAM)

Pick your best model from the comparison above and export bounding boxes.

In [ ]:
from INIA import export_bboxes, get_bboxes

# Use whichever model scored highest Dice
best_arch = results_df['dice_coef'].idxmax()
print(f"Best architecture: {best_arch}")

# Retrain or reuse — if you still have model_pp in memory and UNet++ won:
# For now, let's retrain the best one cleanly
from INIA import fit
best_model, _ = fit(best_arch, X_train, y_train, epochs=50)

# Export bounding boxes
bboxes = export_bboxes(best_model, X_test, output_path="bboxes_for_team2.npy")

In [ ]:
# Quick sanity check — visualize a few bboxes on the images
import matplotlib.patches as patches

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, ax in enumerate(axes):
    ax.imshow(X_test[i].squeeze(), cmap='gray')
    bbox = bboxes[i]
    if bbox[0] != -1:  # not empty
        x_min, y_min, x_max, y_max = bbox
        rect = patches.Rectangle(
            (x_min, y_min), x_max - x_min, y_max - y_min,
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
    ax.set_title(f'BBox: {bbox}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### ✅ Send `bboxes_for_team2.npy` to Team 2
They can load it with `np.load('bboxes_for_team2.npy')` — each row is `[x_min, y_min, x_max, y_max]`.

---
## Step 4 — nnUNet (Research & Setup)

nnUNet is a separate PyTorch-based framework. It **cannot** be added as a simple Keras model.

**Next steps:**
1. Read the nnUNet docs: https://github.com/MIC-DKFZ/nnUNet
2. Check if there's a Colab-friendly version
3. Train it separately on the same dataset
4. Load the predictions and add them to the comparison table

For now, you can add a manual row to your comparison:

In [ ]:
# Placeholder — fill in after running nnUNet separately
import pandas as pd

# Uncomment and fill in once you have nnUNet results:
# nnunet_results = {'dice_coef': 0.0, 'iou_coef': 0.0, 'bin_acc': 0.0, 'precision': 0.0, 'recall': 0.0, 'loss': 0.0}
# full_results = pd.concat([results_df, pd.DataFrame(nnunet_results, index=['nnunet'])])
# full_results

---
## 5. Save Everything

In [ ]:
# Save comparison table
results_df.to_csv('architecture_comparison.csv')
print('Saved architecture_comparison.csv')

# Save best model
best_model.save('best_model.keras')
print(f'Saved best model ({best_arch})')

# Optional: save to Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# best_model.save('/content/drive/MyDrive/IMPACT/best_model.keras')
# results_df.to_csv('/content/drive/MyDrive/IMPACT/architecture_comparison.csv')